# Calcolo Punteggi Amenity per Immobili

Questo notebook calcola i punteggi amenity per gli immobili utilizzando dati POI spaziali.

## Import Required Libraries

Importa le librerie necessarie come json, os, numpy, pandas, sklearn.neighbors e sys.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from scipy.stats import percentileofscore
import sys

## Define Constants

Definisce le costanti come EARTH_RADIUS_KM e MAX_DISTANCE_M.

In [ ]:
# Costanti
EARTH_RADIUS_KM = 6371.0
MAX_DISTANCE_M = 2000  # Distanza massima per considerare POI

In [ ]:
# Definizione delle classi per le amenity
amenity_to_class = {}

# Sanità
breve_sanita = ["pharmacy", "defibrillator"]
medio_sanita = ["doctor", "dentist", "clinic", "physiotherapist", "veterinary", "optometrist", "psychotherapist", "physiotherapist-osteopathy", "alternative"]
lungo_sanita = ["hospital", "laboratory", "dialysis", "hospice", "rehabilitation", "audiologist", "medical_supply", "blood_donation"]

for a in breve_sanita:
    amenity_to_class[a] = 'breve'
for a in medio_sanita:
    amenity_to_class[a] = 'medio'
for a in lungo_sanita:
    amenity_to_class[a] = 'lungo'

# Mobilità
breve_mobilita = ["bus_stop", "tram_stop", "bicycle_parking", "taxi", "subway_entrance", "charging_station"]
medio_mobilita = ["parking", "bicycle_rental", "car_sharing"]
lungo_mobilita = ["station", "car_rental"]

for a in breve_mobilita:
    amenity_to_class[a] = 'breve'
for a in medio_mobilita:
    amenity_to_class[a] = 'medio'
for a in lungo_mobilita:
    amenity_to_class[a] = 'lungo'

# Verde
breve_verde = ["playground", "garden", "grass", "park"]
medio_verde = ["recreation_ground", "scrub"]
lungo_verde = ["forest", "nature_reserve", "wood", "allotments"]

for a in breve_verde:
    amenity_to_class[a] = 'breve'
for a in medio_verde:
    amenity_to_class[a] = 'medio'
for a in lungo_verde:
    amenity_to_class[a] = 'lungo'

# Sport
breve_sport = ["fitness_centre", "pitch"]
medio_sport = ["swimming_pool", "sports_centre", "bowling_alley"]
lungo_sport = ["stadium", "golf_course", "water_park", "ice_rink"]

for a in breve_sport:
    amenity_to_class[a] = 'breve'
for a in medio_sport:
    amenity_to_class[a] = 'medio'
for a in lungo_sport:
    amenity_to_class[a] = 'lungo'

# Commerciale
breve_commerciale = ["supermarket", "bakery", "greengrocer", "newsagent", "kiosk", "convenience", "tobacco", "laundry", "hairdresser", "butcher", "pastry", "ice_cream", "deli", "stationery", "grocery", "vending_machine"]
medio_commerciale = ["clothes", "shoes", "florist", "hardware", "chemist", "pet", "books", "gift", "optician", "beauty", "mobile_phone", "jewelry", "toys", "electronics", "bicycle", "travel_agency", "dry_cleaning", "photo", "video", "confectionery", "alcohol", "wine", "cheese", "dairy", "seafood", "pasta", "spices", "tea", "coffee", "coffee_roasting", "herbalist", "tattoo", "massage", "spa", "copyshop", "shoe_repair", "tailor", "sewing", "fabric", "bag", "fashion_accessories", "cosmetics", "variety_store", "second_hand", "video_games", "sports", "nutrition_supplements", "perfumery", "watches", "baby_goods", "houseware", "repair", "mobile_phone_accessories", "party", "religion", "locksmith", "art", "frame", "ticket", "food", "general", "pet_grooming"]
lungo_commerciale = ["mall", "department_store", "furniture", "car", "car_repair", "car_parts", "motorcycle", "motorcycle_repair", "funeral_directors", "kitchen", "bathroom_furnishing", "bed", "lighting", "curtain", "window_blind", "carpet", "flooring", "tiles", "paint", "doityourself", "garden_centre", "wholesale", "trade", "auction_house", "antiques", "glaziery", "weapons", "hunting", "fishing", "scuba_diving", "boat", "water_sports", "army", "outpost", "erotic", "bookmaker", "money_lender", "pawnbroker", "gold_buyer", "telecommunication", "computer", "printer_ink", "cartridges", "hifi", "musical_instrument", "appliance", "hvac", "security", "tool_hire", "charity", "craft", "leather", "pottery", "model", "hobby", "games", "anime", "wigs", "hearing_aids", "rice", "fair_trade", "printing", "plaques", "scooter", "tyres", "caravan", "brewing_supplies", "country_store", "military_surplus", "hairdresser_supply", "vacant", "gas", "outdoor", "radiotechnics"]

for a in breve_commerciale:
    amenity_to_class[a] = 'breve'
for a in medio_commerciale:
    amenity_to_class[a] = 'medio'
for a in lungo_commerciale:
    amenity_to_class[a] = 'lungo'

# Educazione
breve_educazione = ["kindergarten", "school", "community_centre"]
medio_educazione = ["library", "music_school", "driving_school"]
lungo_educazione = ["university", "college", "museum", "research_institute"]

for a in breve_educazione:
    amenity_to_class[a] = 'breve'
for a in medio_educazione:
    amenity_to_class[a] = 'medio'
for a in lungo_educazione:
    amenity_to_class[a] = 'lungo'

## Load Immobili Data

Carica il dataset degli immobili da un file Parquet utilizzando pandas.

In [ ]:
def load_immobili_data():
    """Carica il dataset degli immobili."""
    # Aggiungi la root del progetto al path
    sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../')))
    from app.core.config import DATASET_FULL
    immobili_path = os.path.join(os.path.dirname('__file__'), '../../', DATASET_FULL)
    immobili_path = os.path.abspath(immobili_path)
    print(f"Caricando immobili da {immobili_path}")
    df = pd.read_parquet(immobili_path)
    print(f"Caricato {len(df)} immobili")
    return df

## Load POIs Data

Carica i dati POI categorizzati da un file JSON.

In [ ]:
def load_pois_data():
    """Carica i POI categorizzati."""
    pois_path = os.path.join(os.path.dirname('__file__'), '../01_pois/pois_by_category.json')
    print(f"Caricando POI da {pois_path}")
    with open(pois_path, 'r', encoding='utf-8') as f:
        pois_by_category = json.load(f)
    print(f"Caricato POI per {len(pois_by_category)} categorie")
    return pois_by_category

## Build Spatial Index

Costruisce un indice spaziale utilizzando BallTree per tutti i POI.

In [ ]:
def build_spatial_index(pois_by_category):
    """Costruisce l'indice spaziale per tutti i POI."""
    all_pois = []
    for category, amenities in pois_by_category.items():
        for amenity, pois in amenities.items():
            for poi in pois:
                all_pois.append({
                    'category': category,
                    'amenity': amenity,
                    'lat': poi['lat'],
                    'lon': poi['lon']
                })

    if not all_pois:
        raise ValueError("Nessun POI trovato")

    # Coordinate in radianti per BallTree
    coords = np.array([[np.radians(p['lat']), np.radians(p['lon'])] for p in all_pois])
    tree = BallTree(coords, metric='haversine')

    print(f"Costruito indice spaziale con {len(all_pois)} POI")
    return tree, all_pois

In [ ]:
# Debug: verifica range coordinate POI
immobili_df = load_immobili_data()
pois_by_category = load_pois_data()
tree, all_pois = build_spatial_index(pois_by_category)

# Verifica range POI
poi_lats = [p['lat'] for p in all_pois]
poi_lons = [p['lon'] for p in all_pois]
print(f"POI - Latitudine: min={min(poi_lats):.6f}, max={max(poi_lats):.6f}")
print(f"POI - Longitudine: min={min(poi_lons):.6f}, max={max(poi_lons):.6f}")

# Verifica range immobili
immobili_lats = immobili_df['latitudine'].values
immobili_lons = immobili_df['longitudine'].values
print(f"\nImmobili - Latitudine: min={immobili_lats.min():.6f}, max={immobili_lats.max():.6f}")
print(f"Immobili - Longitudine: min={immobili_lons.min():.6f}, max={immobili_lons.max():.6f}")

# Verifica se ci sono valori anomali
print(f"\nNumero totale immobili: {len(immobili_df)}")
print(f"Numero totale POI: {len(all_pois)}")

## Calculate Amenity Scores

Calcola i punteggi amenity per ogni immobile utilizzando query spaziali e decadimento esponenziale.

In [ ]:
from scipy.stats import percentileofscore

def calculate_amenity_scores(immobili_df, tree, all_pois, output_path, all_amenities):
    """Calcola i punteggi amenity per ogni immobile e ogni amenity, con score 0 se non presente, e i percentili per amenity."""
    # Definizione degli alpha per le classi
    alpha_breve = 2 * np.log(2)  # Breve
    alpha_medio = np.log(2)      # Medio
    alpha_lungo = 0.5 * np.log(2)  # Lungo

    max_dist_rad = MAX_DISTANCE_M / 1000 / EARTH_RADIUS_KM

    min_dist = float('inf')
    max_dist_val = -float('inf')

    # Dizionari per accumulare i dati per amenity
    amenity_data = {amenity: {'immobile_ids': [], 'scores': []} for amenity in all_amenities}

    for idx in range(len(immobili_df)):
        immobile = immobili_df.iloc[idx]
        immobile_id = immobile['id']
        immobile_lat = immobile['latitudine']
        immobile_lon = immobile['longitudine']

        # Query POI entro MAX_DISTANCE_M con distanze
        query_point = np.array([[np.radians(immobile_lat), np.radians(immobile_lon)]])
        indices_list, distances_list = tree.query_radius(query_point, r=max_dist_rad, return_distance=True)
        indices = np.array(indices_list[0], dtype=int)
        distances = np.array(distances_list[0], dtype=float)
        distances_km = distances * EARTH_RADIUS_KM

        # Aggrega punteggi per amenity per questo immobile
        amenity_scores = {amenity: 0.0 for amenity in all_amenities}

        if indices.size > 0:
            min_dist = min(min_dist, distances.min())
            max_dist_val = max(max_dist_val, distances.max())

            for i, poi_idx in enumerate(indices):
                poi = all_pois[poi_idx]
                classe = amenity_to_class.get(poi['amenity'], 'lungo')
                if classe == 'breve':
                    alpha = alpha_breve
                elif classe == 'medio':
                    alpha = alpha_medio
                else:
                    alpha = alpha_lungo
                score_contrib = np.exp(-alpha * distances_km[i])
                amenity = poi['amenity']
                amenity_scores[amenity] += score_contrib

        # Accumula i dati in memoria
        for amenity in all_amenities:
            amenity_data[amenity]['immobile_ids'].append(immobile_id)
            amenity_data[amenity]['scores'].append(amenity_scores[amenity])

        if (idx + 1) % 100 == 0:
            print(f"Elaborati {idx + 1}/{len(immobili_df)} immobili")

    # Calcola percentili in modo efficiente usando numpy
    print("Calcolo percentili...")
    with open(output_path, 'w') as f:
        f.write('immobile_id,amenity,score,percentile\n')
        
        for amenity in all_amenities:
            immobile_ids = np.array(amenity_data[amenity]['immobile_ids'])
            scores = np.array(amenity_data[amenity]['scores'])
            
            # Calcola percentili in modo vettorizzato
            percentiles = np.zeros(len(scores))
            for i in range(len(scores)):
                percentiles[i] = percentileofscore(scores, scores[i], kind='rank')
            
            # Scrivi i risultati
            for i in range(len(immobile_ids)):
                f.write(f"{int(immobile_ids[i])},{amenity},{scores[i]:.6f},{percentiles[i]:.6f}\n")
            
            print(f"  Completata amenity: {amenity}")

    print("Salvataggio completato!")
    return None, min_dist, max_dist_val

## Save Results

Salva i risultati in un file CSV ed esegue il processo principale.

In [ ]:
def main(subset_size=None):
    print("Inizio calcolo immobili_amenity_scores_aggregated.csv")

    # Carica dati
    immobili_df = load_immobili_data()

    # Usa subset se richiesto
    if subset_size:
        immobili_df = immobili_df.head(subset_size)
        print(f"DEBUG: Usando solo {len(immobili_df)} immobili per test")
        output_path = 'immobili_amenity_scores_aggregated_debug.csv'
    else:
        output_path = 'immobili_amenity_scores_aggregated.csv'

    pois_by_category = load_pois_data()

    # Raccogli tutte le amenity uniche
    all_amenities = set()
    for category, amenities in pois_by_category.items():
        for amenity in amenities.keys():
            all_amenities.add(amenity)
    all_amenities = sorted(list(all_amenities))

    # Costruisci indice spaziale
    tree, all_pois = build_spatial_index(pois_by_category)

    # Calcola punteggi
    _, min_dist, max_dist = calculate_amenity_scores(immobili_df, tree, all_pois, output_path, all_amenities)

    print(f"\nStatistiche distanze:")
    print(f"  Min distanza: {min_dist * EARTH_RADIUS_KM * 1000:.2f} m")
    print(f"  Max distanza: {max_dist * EARTH_RADIUS_KM * 1000:.2f} m")

    print(f"\nSalvato {output_path} (struttura: immobile_id,amenity,score,percentile)")
    print("Completato!")

# Esegui il main
main()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2, 100)
alpha1 = 2 * np.log(2)
alpha2 = np.log(2)
alpha3 = 0.5 * np.log(2)

y1 = np.exp(-alpha1 * x)
y2 = np.exp(-alpha2 * x)
y3 = np.exp(-alpha3 * x)

plt.figure()
plt.plot(x, y1, label=f'alpha = {alpha1:.3f}')
plt.plot(x, y2, label=f'alpha = {alpha2:.3f}')
plt.plot(x, y3, label=f'alpha = {alpha3:.3f}')
plt.xlim(0, 2)
plt.ylim(0, 1)
plt.xlabel('Distanza (km)')
plt.ylabel('Score')
plt.legend()
plt.show()